# CDSH-UHCL
This notebook implements a PyTorch version of the Unified Hierarchical Contrastive Loss (UHCL) for robust representation learning. The workflow combines hierarchical contrastive learning, anomaly detection, and noise-aware separation in a single formulation.

## Overview
The data distribution is treated as a broader data space, where the target inlier context is a subset of a larger universe that also contains outliers and noise. The method builds a hierarchy of semantic structure from noisy to inlier observations:

Noise -> Outlier -> Inlier -> Positive Inlier

The objective encourages similarity to remain structured as samples move through this hierarchy while preserving separation between inliers, outliers, and noise.

## Repository context
This repo is named `CDSH-UHCL` to reflect the contrastive-discriminative hierarchical learning setup used in the implementation.


In [67]:
# evaluate_summary
import os
import random
from typing import List, Tuple, Dict
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from sklearn.metrics import roc_auc_score
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from collections import defaultdict


# Configuration
Each experiment parameter is defined here to make it easy to modify the training setup and evaluation settings.

In [68]:
config = {
    "experiment_name": "standardcifar100vsboth",
    "inlier_dataset": "cifar100",
    "outlier_dataset": "both",
    "epochs": 100,
    "batch_size": 100,
    "anchors_per_fine": 1,
    "lr": 1e-4,
    "min_lr": 1e-6,
    "aux_weight": 0.000,
    "weight_decay": 0.000,
    "alpha": 1/2,
    "alpha_decay": 1/2,
    "embed_dim": 128,
    "num_workers": 0,
    "pretrained": True,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "ckpt_dir": "./content/checkpoints",
    "resume_ckpt_path": "./content/checkpoints/standardcifar100vsboth_last.pth",
}

os.makedirs(config["ckpt_dir"], exist_ok=True)

def set_seed(seed=59):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

class ToTensorSafe:
    def __call__(self, pic):
        if isinstance(pic, torch.Tensor):
            return pic.float() / 255. if pic.max() > 1 else pic.float()
        return transforms.functional.to_tensor(pic)


In [69]:
config.update({
    "noise_trajectory_steps": 50,
    "noise_trajectory_samples": 2000,
    "noise_trajectory_seed": 35,
    "noise_trajectory_final_alpha_bar": 0.0,
    "results_dir": "./results",
})


# Data loaders
This section defines image transformations and helper datasets used for inlier, outlier, and noise sampling.

In [70]:
IMG_SIZE = {"mnist": 28, "fashionmnist": 28, "emnist": 28, "cifar10": 32, "cifar100": 32, "svhn": 32}

def get_transforms(name, mode="train"):
    size = IMG_SIZE.get(name, 32)
    tensorizer = ToTensorSafe()
    if name in ["mnist", "fashionmnist", "emnist"]:
        if mode == "train":
            return transforms.Compose([tensorizer])
        else:
            return transforms.Compose([transforms.Resize(size), tensorizer])

    if mode == "train":
        return transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(size),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

class RandomNoiseDataset(Dataset):
    def __init__(self, num: int, shape: Tuple[int, int, int]):
        self.num = num
        self.shape = shape
    def __len__(self):
        return self.num
    def __getitem__(self, idx):
        return torch.rand(self.shape), -1


In [71]:
import copy
import random
import numpy as np
import torch
from torch.utils.data import Subset

def get_random_coarse_split(dataset, split_ratio=0.5, seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    all_coarse_labels = list(dataset.coarse_label_indices.keys())
    random.shuffle(all_coarse_labels)
    n_inliers = int(len(all_coarse_labels) * split_ratio)
    inlier_labels = set(all_coarse_labels[:n_inliers])
    print(f"Split generated: {len(inlier_labels)} Inlier Coarse Classes")
    return inlier_labels

def split_dataset_by_labels(full_dataset, inlier_coarse_labels):
    outlier_indices = []
    dataset_coarse_keys = full_dataset.coarse_label_indices.keys()
    for c_lbl in dataset_coarse_keys:
        if c_lbl not in inlier_coarse_labels:
            outlier_indices.extend(full_dataset.coarse_label_indices[c_lbl])
    out_dataset = Subset(full_dataset, outlier_indices)

    in_dataset = copy.copy(full_dataset)
    in_dataset.fine_label_indices = copy.deepcopy(full_dataset.fine_label_indices)
    in_dataset.coarse_label_indices = copy.deepcopy(full_dataset.coarse_label_indices)

    in_dataset.coarse_label_indices = {
        c: idxs for c, idxs in in_dataset.coarse_label_indices.items()
        if c in inlier_coarse_labels
    }

    valid_fine_labels = []
    for fine_lbl, indices in in_dataset.fine_label_indices.items():
        if not indices:
            continue
        _, _, sample_coarse = full_dataset[indices[0]]
        if sample_coarse in inlier_coarse_labels:
            valid_fine_labels.append(fine_lbl)

    in_dataset.fine_label_indices = {
        f: in_dataset.fine_label_indices[f] for f in valid_fine_labels
    }

    return in_dataset, out_dataset


In [72]:
import torch
from torchvision import datasets
import numpy as np
from collections import defaultdict

class CIFAR100Hierarchy(datasets.CIFAR100):
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform, target_transform=target_transform, download=download)
        self.coarse_to_fine = {
            0: [4, 30, 55, 72, 95],
            1: [1, 32, 67, 73, 91],
            2: [54, 62, 70, 82, 92],
            3: [9, 10, 16, 28, 61],
            4: [0, 51, 53, 57, 83],
            5: [22, 39, 40, 86, 87],
            6: [5, 20, 25, 84, 94],
            7: [6, 7, 14, 18, 24],
            8: [3, 42, 43, 88, 97],
            9: [12, 17, 37, 68, 76],
            10: [23, 33, 49, 60, 71],
            11: [15, 19, 21, 31, 38],
            12: [34, 63, 64, 66, 75],
            13: [26, 45, 77, 79, 99],
            14: [2, 11, 35, 46, 98],
            15: [27, 29, 44, 78, 93],
            16: [36, 50, 65, 74, 80],
            17: [47, 52, 56, 59, 96],
            18: [8, 13, 48, 58, 90],
            19: [41, 69, 81, 85, 89],
        }
        self.fine_to_coarse = {}
        for coarse_label, fine_labels in self.coarse_to_fine.items():
            for fine_label in fine_labels:
                self.fine_to_coarse[fine_label] = coarse_label

        self.coarse_targets = [self.fine_to_coarse[fine_label] for fine_label in self.targets]
        self.fine_label_indices = defaultdict(list)
        self.coarse_label_indices = defaultdict(list)
        for i, (fine_label, coarse_label) in enumerate(zip(self.targets, self.coarse_targets)):
            self.fine_label_indices[fine_label].append(i)
            self.coarse_label_indices[coarse_label].append(i)

    def __getitem__(self, index):
        img, fine_target = super().__getitem__(index)
        coarse_target = self.coarse_targets[index]
        return img, fine_target, coarse_target

class CIFAR10Indexed(datasets.CIFAR10):
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super().__init__(root, train=train, transform=transform, target_transform=target_transform, download=download)
        self.label_indices = defaultdict(list)
        for i, label in enumerate(self.targets):
            self.label_indices[label].append(i)


In [73]:
def sample_hierarchy_batch(in_dataset, out_dataset, noise_dataset,
                           batch_size, device, N=4, is_hierarchical=False,
                           anchors_per_fine=config["anchors_per_fine"]):
    X_batch = []
    yfine_batch = []
    ycoarse_batch = []
    y_batch = []

    if is_hierarchical:
        fine_labels = list(in_dataset.fine_label_indices.keys())
        random.shuffle(fine_labels)
        for fine_label in fine_labels:
            fine_indices = in_dataset.fine_label_indices[fine_label]
            if len(fine_indices) < anchors_per_fine:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=True)
            else:
                anchors = np.random.choice(fine_indices, anchors_per_fine, replace=False)

            for idx in anchors:
                try:
                    x0, fine0, coarse0 = in_dataset[idx]
                    pos_candidates = [i for i in in_dataset.fine_label_indices[fine0] if i != idx]
                    pos_idx = random.choice(pos_candidates)
                    x1, _, _ = in_dataset[pos_idx]

                    coarse_candidates = [
                        i for i in in_dataset.coarse_label_indices[coarse0]
                        if in_dataset.targets[i] != fine0
                    ]
                    neg_fine_idx = random.choice(coarse_candidates)
                    x2, fine2, _ = in_dataset[neg_fine_idx]

                    all_coarse = list(in_dataset.coarse_label_indices.keys())
                    diff_coarse = [c for c in all_coarse if c != coarse0]
                    rand_coarse = random.choice(diff_coarse)
                    diff_coarse_candidates = in_dataset.coarse_label_indices[rand_coarse]
                    diff_coarse_idx = random.choice(diff_coarse_candidates)
                    x3, fine3, coarse3 = in_dataset[diff_coarse_idx]

                    x4, *rest = out_dataset[np.random.randint(len(out_dataset))]
                    x5, _ = noise_dataset[np.random.randint(len(noise_dataset))]

                    X_tuple = torch.stack([x0, x1, x2, x3, x4, x5])
                    yfine_tuple = [fine0, fine0, fine2, fine3]
                    ycoarse_tuple = [coarse0, coarse0, coarse0, coarse3]

                    X_batch.append(X_tuple)
                    yfine_batch.append(yfine_tuple)
                    ycoarse_batch.append(ycoarse_tuple)
                except Exception:
                    x0, _, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))
    else:
        labels = list(in_dataset.label_indices.keys())
        random.shuffle(labels)
        for label in labels:
            label_indices = in_dataset.label_indices[label]
            if len(label_indices) < anchors_per_fine:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=True)
            else:
                anchors = np.random.choice(label_indices, anchors_per_fine * 10, replace=False)
            for idx in anchors:
                try:
                    x0, y0 = in_dataset[idx]
                    pos_candidates = [i for i in in_dataset.label_indices[y0] if i != idx]
                    pos_idx = random.choice(pos_candidates) if pos_candidates else idx
                    x1, _ = in_dataset[pos_idx]
                    diff_labels = [l for l in labels if l != y0]
                    neg_label = random.choice(diff_labels)
                    neg_idx = random.choice(in_dataset.label_indices[neg_label])
                    x2, y2 = in_dataset[neg_idx]
                    x3, _ = out_dataset[np.random.randint(len(out_dataset))]
                    x4, _ = noise_dataset[np.random.randint(len(noise_dataset))]
                    X_tuple = torch.stack([x0, x1, x2, x3, x4])
                    y_tuple = [y0, y0, y2]
                    X_batch.append(X_tuple)
                    y_batch.append(y_tuple)
                except Exception:
                    x0, _ = in_dataset[idx]
                    fallback = [x0] * N
                    X_batch.append(torch.stack(fallback))

        yfine_batch = y_batch
        ycoarse_batch = None

    if len(X_batch) > batch_size:
        X_batch = X_batch[:batch_size]
        if yfine_batch is not None:
            yfine_batch = yfine_batch[:batch_size]
        if ycoarse_batch is not None:
            ycoarse_batch = ycoarse_batch[:batch_size]

    X_batch = torch.stack(X_batch).to(device)

    if ycoarse_batch is None:
        ycoarse_batch = torch.zeros((len(yfine_batch), 1), dtype=torch.long)

    return (
        X_batch,
        torch.tensor(yfine_batch, dtype=torch.long, device=device),
        torch.tensor(ycoarse_batch, dtype=torch.long, device=device),
    )


# Model and projection
This section defines the backbone and the projection head used to map image features into the embedding space for UHCL.

In [74]:
class SmallResNet(nn.Module):
    def __init__(self, embed_dim=128, pretrained=False, in_channels=3, depth=18, num_classes=None):
        super().__init__()
        if depth == 50:
            net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 2048
        else:
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            proj_in_features = 512

        if in_channels != 3:
            original_conv1 = net.conv1
            net.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
            if pretrained:
                net.conv1.weight.data[:, :3, :, :] = original_conv1.weight.data
                nn.init.kaiming_normal_(net.conv1.weight.data[:, 3:, :, :], mode='fan_out', nonlinearity='relu')

        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.projector = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(proj_in_features, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embed_dim),
        )

        self.clf_head = None
        if num_classes is not None:
            self.clf_head = nn.Linear(embed_dim, num_classes)

    def forward_features_and_logits(self, x):
        emb = self(x, return_embeddings=True)
        if self.clf_head is None:
            raise ValueError("clf_head is not defined. Make sure num_classes was set.")
        logits = self.clf_head(emb)
        return emb, logits

    def forward(self, x, return_embeddings=True):
        z = self.backbone(x)
        z = self.projector(z)
        emb = F.normalize(z, p=2, dim=1)
        return emb, z


# UHCL objective
The loss enforces hierarchical similarity ordering between inlier, outlier, and noise samples.

In [75]:
import torch
import torch.nn.functional as F

def uhcl(emb, N=5, alpha_base=0.5, alpha_decay=0.5, use_identity=False, device=None):
    if device is None:
        device = emb.device

    B, N, D = emb.shape
    assert N >= 3, f"UHCL requires at least 3 hierarchy levels, got {N}"

    S = torch.bmm(emb, emb.transpose(1, 2))
    total_loss = torch.tensor(0.0, device=device)

    noise_sim = S[:, 0, N - 1]
    total_loss += F.relu(noise_sim).mean()

    alpha_by_level = {}
    for j in range(1, N):
        depth = (N - 1) - j
        alpha_by_level[j] = alpha_base * (alpha_decay ** depth)

    margin_sum = sum(alpha_by_level.values())
    if margin_sum > 1.0:
        scale = 1.0 / margin_sum
        for j in alpha_by_level:
            alpha_by_level[j] *= scale

    hierarchy_loss = torch.tensor(0.0, device=device)
    for j in range(1, N):
        alpha_j = alpha_by_level[j]
        level_loss = torch.tensor(0.0, device=device)

        for i in range(0, j):
            partial_loss = torch.tensor(0.0, device=device)
            for k in range(i, j):
                positive_sim = S[:, i, j - 1]
                negative_sim = S[:, k, j]
                hinge = F.relu(negative_sim - positive_sim + alpha_j)
                partial_loss += hinge.mean()
            num_terms = j - i
            lambda_ij = 1.0 / num_terms
            level_loss += partial_loss * lambda_ij

        lambda_j = 1.0 / j
        hierarchy_loss += level_loss * lambda_j

    total_loss += hierarchy_loss
    return total_loss


# Build experiment
This section assembles the datasets, noise sampler, and model used in the UHCL training pipeline.

In [76]:
def build_experiment(cfg):
    device = cfg["device"]
    ttr = get_transforms(cfg["inlier_dataset"], "train")
    tte = get_transforms(cfg["inlier_dataset"], "test")

    if cfg["inlier_dataset"] == "cifar100":
        in_train = CIFAR100Hierarchy("./data", train=True, transform=ttr, download=True)
        in_test = CIFAR100Hierarchy("./data", train=False, transform=tte, download=True)
        is_hierarchical = True
        N_levels_uhcl = 6
        depth = 50

    elif cfg["inlier_dataset"] == "cifar10":
        in_train = CIFAR10Indexed("./data", train=True, transform=ttr, download=True)
        in_test = CIFAR10Indexed("./data", train=False, transform=tte, download=True)
        is_hierarchical = False
        N_levels_uhcl = 5
        depth = 18
    else:
        raise ValueError(f"Unsupported inlier_dataset: {cfg['inlier_dataset']}")

    embed_dim = cfg["embed_dim"]

    print(f"Training OOD dataset: {cfg['outlier_dataset']}")

    if cfg["outlier_dataset"] == "svhn":
        out_train = datasets.SVHN("./data", split="train", transform=ttr, download=True)
        out_test = datasets.SVHN("./data", split="test", transform=tte, download=True)
    elif cfg["outlier_dataset"] == "cifar10":
        out_train = datasets.CIFAR10("./data", train=True, transform=ttr, download=True)
        out_test = datasets.CIFAR10("./data", train=False, transform=tte, download=True)
    elif cfg["outlier_dataset"] == "both":
        cifar10_train = datasets.CIFAR10("./data", train=True, transform=ttr, download=True)
        cifar10_test = datasets.CIFAR10("./data", train=False, transform=tte, download=True)
        svhn_train = datasets.SVHN("./data", split="train", transform=ttr, download=True)
        svhn_test = datasets.SVHN("./data", split="test", transform=tte, download=True)

        n = min(len(cifar10_train), len(svhn_train))
        rng = np.random.default_rng(42)

        cifar10_idx = rng.choice(len(cifar10_train), size=n, replace=False)
        svhn_idx = rng.choice(len(svhn_train), size=n, replace=False)

        cifar10_train = torch.utils.data.Subset(cifar10_train, cifar10_idx)
        svhn_train = torch.utils.data.Subset(svhn_train, svhn_idx)

        out_train = torch.utils.data.ConcatDataset([cifar10_train, svhn_train])
        out_test = torch.utils.data.ConcatDataset([cifar10_test, svhn_test])

    elif cfg["outlier_dataset"] == "cifar100":
        out_train = CIFAR100Hierarchy("./data", train=True, transform=ttr, download=True)
        out_test = CIFAR100Hierarchy("./data", train=False, transform=tte, download=True)
    else:
        raise ValueError(f"Unsupported outlier_dataset: {cfg['outlier_dataset']}")

    in_channels = 1 if cfg["inlier_dataset"] in ["mnist", "fashionmnist", "emnist"] else 3
    image_size = IMG_SIZE.get(cfg["inlier_dataset"], 32)

    noise_train = RandomNoiseDataset(len(in_train), (in_channels, image_size, image_size))
    noise_test = RandomNoiseDataset(len(in_test), (in_channels, image_size, image_size))

    print(f"Initializing SmallResNet with pretrained={cfg['pretrained']}")
    model = SmallResNet(embed_dim=embed_dim, in_channels=in_channels, pretrained=cfg["pretrained"], depth=depth, num_classes=3).to(device)

    if cfg["inlier_dataset"] == "cifar100":
        model.coarse_head = nn.Linear(embed_dim, 20).to(device)
        model.fine_head = nn.Linear(embed_dim, 100).to(device)
    elif cfg["inlier_dataset"] == "cifar10":
        model.inlier_head = nn.Linear(embed_dim, 10).to(device)

    return {
        "model": model,
        "in_train": in_train,
        "in_test": in_test,
        "out_train": out_train,
        "out_test": out_test,
        "noise_train": noise_train,
        "noise_test": noise_test,
        "is_hierarchical": is_hierarchical,
        "N_levels_uhcl": N_levels_uhcl,
        "embed_dim": embed_dim,
    }


# Training
This section trains the model using UHCL and optional auxiliary classification heads.

In [77]:
def train(cfg):
    device = cfg["device"]
    exp = build_experiment(cfg)
    model = exp["model"]
    in_train = exp["in_train"]
    out_train = exp["out_train"]
    noise_train = exp["noise_train"]

    N_levels_uhcl = exp["N_levels_uhcl"]
    is_hierarchical = exp["is_hierarchical"]
    embed_dim = exp["embed_dim"]

    resume_path = cfg.get("resume_ckpt_path", None)
    if resume_path is not None:
        if not os.path.exists(resume_path):
            raise FileNotFoundError(f"Checkpoint not found:\n{resume_path}")
        model.load_state_dict(torch.load(resume_path, map_location=device))

    epochs = cfg["epochs"]
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=cfg["min_lr"])

    main_loader = DataLoader(in_train, batch_size=cfg["batch_size"], shuffle=True, num_workers=cfg["num_workers"])

    for ep in range(epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0
        pbar = tqdm(main_loader, desc=f"Epoch {ep + 1}/{epochs}")

        for _ in pbar:
            X, y_fine, y_coarse = sample_hierarchy_batch(
                in_train, out_train, noise_train, batch_size=cfg["batch_size"], device=device,
                N=N_levels_uhcl, is_hierarchical=is_hierarchical
            )

            if X.size(0) == 0:
                continue

            B, N_levels, C, H, W = X.shape
            X = X.view(B * N_levels, C, H, W)
            Z_all, logits_all = model(X)
            Z_all = Z_all.view(B, N_levels, -1)
            logits_all = logits_all.view(B, N_levels, -1)

            uhcl_loss = uhcl(Z_all, N=N_levels_uhcl, alpha_base=cfg["alpha"], alpha_decay=cfg["alpha_decay"], device=device)

            if cfg["inlier_dataset"] == "cifar100":
                inlier_embs = logits_all[:, :4, :].reshape(-1, embed_dim)
                fine_labels = y_fine[:, :4].reshape(-1).long()
                coarse_labels = y_coarse[:, :4].reshape(-1).long()
                fine_logits = model.fine_head(inlier_embs)
                coarse_logits = model.coarse_head(inlier_embs)
                fine_loss = F.cross_entropy(fine_logits, fine_labels)
                coarse_loss = F.cross_entropy(coarse_logits, coarse_labels)
                aux_loss = fine_loss + coarse_loss
            else:
                inlier_embs = logits_all[:, :3, :].reshape(-1, embed_dim)
                class_labels = y_fine[:, :3].reshape(-1).long()
                inlier_logits = model.inlier_head(inlier_embs)
                aux_loss = F.cross_entropy(inlier_logits, class_labels)

            total_loss = uhcl_loss + cfg["aux_weight"] * aux_loss
            opt.zero_grad()
            total_loss.backward()
            opt.step()

            epoch_loss += uhcl_loss.item()
            batch_count += 1

            pbar.set_postfix({"UHCL": f"{uhcl_loss.item():.4f}", "Aux": f"{aux_loss.item():.4f}"})

        scheduler.step()
        avg_loss = epoch_loss / batch_count if batch_count > 0 else float("nan")
        current_lr = opt.param_groups[0]["lr"]
        print(f"Epoch {ep + 1}/{epochs} - UHCL: {avg_loss:.6f} - LR: {current_lr:.6e}")

        if (ep + 1) % 10 == 0:
            os.makedirs(cfg["ckpt_dir"], exist_ok=True)
            path = os.path.join(cfg["ckpt_dir"], f"{cfg['experiment_name']}_ep{ep + 1}.pth")
            torch.save(model.state_dict(), path)
            print(f"Checkpoint saved: {path}")

    os.makedirs(cfg["ckpt_dir"], exist_ok=True)
    final_path = os.path.join(cfg["ckpt_dir"], f"{cfg['experiment_name']}_last.pth")
    torch.save(model.state_dict(), final_path)
    print("\nTraining complete.")
    print(f"Final checkpoint: {final_path}")
    model.eval()
    return {
        "model": model,
        "in_train": in_train,
        "in_test": exp["in_test"],
        "out_train": out_train,
        "out_test": exp["out_test"],
        "noise_train": noise_train,
        "noise_test": exp["noise_test"],
    }


# Evaluation
This section evaluates representation quality, inlier classification, OOD detection, and noise sensitivity.

In [78]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, roc_curve

def _classification_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def _normalize_np(x):
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms

@torch.no_grad()
def extract_eval_features(dataset, model, device, batch_size=256, hierarchical=False):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    all_z = []
    all_head_features = []
    all_fine_labels = []
    all_coarse_labels = []

    for batch in loader:
        images = batch[0]
        fine_labels = batch[1] if len(batch) >= 2 else None
        coarse_labels = batch[2] if hierarchical and len(batch) >= 3 else None
        images = images.to(device)
        z, head_features = model(images)
        all_z.append(z.detach().cpu())
        all_head_features.append(head_features.detach().cpu())
        if fine_labels is not None:
            if not torch.is_tensor(fine_labels):
                fine_labels = torch.as_tensor(fine_labels)
            all_fine_labels.append(fine_labels.detach().cpu().long())
        if coarse_labels is not None:
            if not torch.is_tensor(coarse_labels):
                coarse_labels = torch.as_tensor(coarse_labels)
            all_coarse_labels.append(coarse_labels.detach().cpu().long())

    z = torch.cat(all_z, dim=0).numpy()
    head_features = torch.cat(all_head_features, dim=0).numpy()
    fine_labels = torch.cat(all_fine_labels, dim=0).numpy() if all_fine_labels else None
    coarse_labels = torch.cat(all_coarse_labels, dim=0).numpy() if all_coarse_labels else None
    return z, head_features, fine_labels, coarse_labels


The notebook continues with full evaluation utilities, plotting, and the manuscript-oriented analysis pipeline. The complete implementation is retained in the original notebook structure for reproducible experimentation.